# Offline preprocess (shared)

No W&B required. Run bootstrap cell first.

## 0) Bootstrap (Kaggle / Colab)

Run once per session: clone or update this repo, `cd` into it, install deps.

**Enable Internet** in the Kaggle notebook settings, or the clone and pip install fail.

`REPO_BRANCH` must point at the branch that holds the preprocessing code. Once
`feat/preprocessing` is merged into `main` you can set it back to `"main"`.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/tuan8p/VN-Traffic-Sign-Classification.git"
REPO_DIR = "VN-Traffic-Sign-Classification"
REPO_BRANCH = "feat/preprocessing"   # <- the preprocessing code lives here, not on main

base = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
repo_path = base / REPO_DIR


def sh(cmd: str, check: bool = True) -> int:
    print(">>", cmd)
    return subprocess.run(cmd, shell=True, check=check).returncode


if (Path.cwd() / ".git").exists() and (Path.cwd() / "configs" / "shared.yaml").exists():
    repo_path = Path.cwd()
    print("using the repo this notebook already sits in:", repo_path)
elif not (repo_path / ".git").exists():
    sh(f'git clone --branch {REPO_BRANCH} {REPO_URL} "{repo_path}"')

# Always land on REPO_BRANCH: a previous run may have cloned the default branch,
# which is what produces "cannot import name find_dataset_root".
git = f'git -C "{repo_path}"'
sh(f"{git} fetch origin {REPO_BRANCH}")
sh(f"{git} checkout -B {REPO_BRANCH} origin/{REPO_BRANCH}")

os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
print("cwd    :", Path.cwd())
sh(f"{git} log --oneline -1")

req = repo_path / "requirements-kaggle.txt"
sh(f'pip install -q -r "{req if req.exists() else repo_path / "requirements.txt"}"')
sh("pip install -q -e .")

# Fail loudly here rather than three cells later.
import importlib
for mod in ("vn_tsc.data.discover", "vn_tsc.data.crops", "vn_tsc.data.grouping",
            "vn_tsc.data.splitting", "vn_tsc.data.augment", "vn_tsc.data.classes"):
    importlib.reload(importlib.import_module(mod)) if mod in sys.modules else importlib.import_module(mod)
from vn_tsc.data.discover import find_dataset_root  # noqa: F401
print("bootstrap OK — preprocessing modules import cleanly")

## 1) Point at the dataset

On Kaggle: **Add Input -> Datasets -> `maitam/vietnamese-traffic-signs`**, which mounts
at `/kaggle/input/vietnamese-traffic-signs`. Locally, set `DATA_ROOT` in `.env` or edit
the list below. Sub-folders are searched, so a wrapper directory inside the archive is fine.

In [ ]:
import os
from pathlib import Path
from vn_tsc.config.resolve import resolve_config
from vn_tsc.data.discover import find_dataset_root

CANDIDATES = [
    os.environ.get("DATA_ROOT"),
    "/kaggle/input/vietnamese-traffic-signs",
    "../dataset",
    "data/raw",
]
DATA_ROOT = next((c for c in CANDIDATES if c and Path(c).exists()), None)
assert DATA_ROOT, f"no dataset found, tried: {CANDIDATES}"
print("DATA_ROOT  :", DATA_ROOT)
print("resolved to:", find_dataset_root(DATA_ROOT))

cfg = resolve_config(shared_yaml="configs/shared.yaml", runtime_yaml="configs/runtime/local.yaml")
OUT = "data/processed"
print("crop size :", cfg["preprocess"]["image_size"],
      "| context margin", cfg["preprocess"]["bbox_crop"]["context_margin"])
print("split     :", cfg["split"]["strategy"], cfg["split"]["ratios"])
print("augment   :", cfg["augment"]["enabled"], "| target", cfg["augment"]["target_count"])

## 2) Run the shared offline preprocess

Every YOLO bounding box becomes one 224x224 classification sample. Near-duplicate
video frames are grouped so they cannot straddle a split, the split is 70/15/15 by
group, and the crop cache plus the shared classical features are written.
Expect ~2-4 minutes and ~2.2 GB.

For the ablation, rebuild with `cfg["augment"]["enabled"] = False` into a different
output folder.

In [ ]:
from vn_tsc.data.preprocess_offline import run_offline_preprocess
from vn_tsc.features.classical import build_feature_store

run_offline_preprocess(cfg, DATA_ROOT, OUT)   # crops + grouping + split + augmentation
build_feature_store(cfg, OUT)                 # HOG / LBP / colour for SVM + Boosting

## 3) Read the report

`preprocess_report.json` holds every number the write-up needs. The leakage block
compares the split shipped with the dataset against ours.

In [ ]:
import json
rep = json.loads(Path(OUT, "preprocess_report.json").read_text(encoding="utf-8"))

d = rep["discovery"]
print("boxes in labels       :", d["n_boxes"])
print("empty label files     :", d["n_empty_labels"])
print("malformed lines       :", d["n_malformed_lines"])
print("boxes clipped to frame:", d["n_clipped_boxes"])
print("dropped < min_box_px  :", rep["bbox_crop"]["dropped_below_min_box_px"])
print()
g = rep["grouping"]
print("near-duplicate clusters:", g["n_groups"], "from", g["n_items"], "frames",
      "| largest", g["largest_group"])
print("LEAKAGE, dataset split :", rep["leakage"]["provided_by_authors"]["n_affected_images"], "images")
print("LEAKAGE, our split     :", rep["leakage"]["ours"]["n_affected_images"], "images")
print()
s = rep["split"]
print("ratios achieved :", s["ratios_achieved"])
print("crops per split :", s["crops_per_split"])
print("classes missing :", s["classes_missing_per_split"])
print()
a = rep["augmentation"]
print(f"augmentation: {a['train_before']} -> {a['train_after']} train crops, "
      f"imbalance {a['imbalance_ratio_before']}:1 -> {a['imbalance_ratio_after']}:1")
print("signs smaller than 32x32 px:",
      f"{100 * rep['crop_size_stats']['share_below_32x32_px']:.0f}%")

## 4) Sanity check

Confirms `metadata.csv` lines up with the arrays, that no source image or
duplicate-cluster spans two splits, and shows real crops so labels can be eyeballed.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from vn_tsc.data.dataset import load_split, load_features, load_class_table

meta = pd.read_csv(Path(OUT, "metadata.csv"))
table = load_class_table(OUT)

for split in ("train", "val", "test", "train_aug"):
    sub = meta[meta.split == split].sort_values("array_index")
    X, y = load_split(split, OUT) if split != "train_aug" else (
        np.load(Path(OUT, "images/X_train_aug.npy"), mmap_mode="r"),
        np.load(Path(OUT, "images/y_train_aug.npy")))
    assert len(sub) == len(X) == len(y), split
    assert (sub.class_id.to_numpy() == y).all(), split
    print(f"{split:9s} {X.shape}  labels match")

real = meta[meta.split != "train_aug"]
for col in ("source_image", "group_id"):
    assert int((real.groupby(col).split.nunique() > 1).sum()) == 0, col
print("no source image and no duplicate-cluster spans two splits")

F, _ = load_features("train", OUT)
print("features:", F.shape)

Xtr, _ = load_split("train", OUT)
tr = meta[meta.split == "train"]
shown = [c for c in table.ids[:16] if len(tr[tr.class_id == c])]
fig, axes = plt.subplots(2, 8, figsize=(16, 4.6))
for ax, c in zip(axes.ravel(), shown):
    idx = int(tr[tr.class_id == c].nlargest(1, "box_area_px").array_index.iloc[0])
    ax.imshow(np.asarray(Xtr[idx]))
    ax.set_title(table[c].label, fontsize=7)
    ax.axis("off")
for ax in axes.ravel()[len(shown):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

## 5) Kaggle: publish the store once, reuse it everywhere

Re-running preprocessing inside every training notebook burns GPU quota, and a
second run is a second chance to end up with a different split. Save `data/processed`
once via **Notebook output -> Create Dataset**, then attach that dataset in
`02_train_svm` / `03_train_boosting` / `04_train_dl` and point `PROCESSED` at it.

In [ ]:
total = 0
for p in sorted(Path(OUT).rglob("*")):
    if p.is_file():
        total += p.stat().st_size
        print(f"{p.stat().st_size / 1e6:9.1f} MB  {p.relative_to(OUT)}")
print(f"\nTOTAL {total / 1e9:.2f} GB")